# Model Training V2
Enhanced model with ELO ratings and player strength features

In [1]:
import pandas as pd
import numpy as np
import os
import pickle
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
import xgboost as xgb

BASE = r'C:\Project\FIFA_World_Cup_2026'

features_df = pd.read_csv(os.path.join(BASE, 'Feature_Engineering', 'features_v2.csv'))

print("Shape:", features_df.shape)
print("Columns:", features_df.columns.tolist())

Shape: (8068, 27)
Columns: ['date', 'home_team', 'away_team', 'home_form', 'away_form', 'form_diff', 'home_avg_scored', 'home_avg_conceded', 'away_avg_scored', 'away_avg_conceded', 'goal_diff', 'home_elo', 'away_elo', 'elo_diff', 'home_elo_winrate', 'away_elo_winrate', 'home_attack', 'away_attack', 'attack_diff', 'home_defense', 'away_defense', 'defense_diff', 'home_mid', 'away_mid', 'is_neutral', 'weight', 'result']


## Prepare Training Data

In [2]:
feature_cols = [
    'home_form', 'away_form', 'form_diff',
    'home_avg_scored', 'home_avg_conceded',
    'away_avg_scored', 'away_avg_conceded',
    'goal_diff', 'is_neutral',
    'home_elo', 'away_elo', 'elo_diff',
    'home_elo_winrate', 'away_elo_winrate',
    'home_attack', 'away_attack', 'attack_diff',
    'home_defense', 'away_defense', 'defense_diff',
    'home_mid', 'away_mid'
]

X = features_df[feature_cols]
y = features_df['result']
weights = features_df['weight']

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, weights, test_size=0.2, random_state=42
)

# Encode labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# Scale for logistic regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)
print("Features used:", len(feature_cols))

Training set: (6454, 22)
Testing set: (1614, 22)
Features used: 22


## Train Logistic Regression V2

In [3]:
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train_enc, sample_weight=w_train)

lr_preds = lr_model.predict(X_test_scaled)
lr_accuracy = accuracy_score(y_test_enc, lr_preds)

print(f"Logistic Regression V2 Accuracy: {lr_accuracy:.4f}")

Logistic Regression V2 Accuracy: 0.5124


## Train XGBoost V2

In [5]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss'
)

xgb_model.fit(X_train, y_train_enc, sample_weight=w_train)

xgb_preds = xgb_model.predict(X_test)
xgb_accuracy = accuracy_score(y_test_enc, xgb_preds)

print(f"XGBoost V2 Accuracy: {xgb_accuracy:.4f}")

XGBoost V2 Accuracy: 0.5483


## V1 vs V2 Comparison

In [6]:
print("=" * 40)
print("      V1 vs V2 ACCURACY COMPARISON")
print("=" * 40)
print(f"V1 Logistic Regression : 0.5211")
print(f"V2 Logistic Regression : {lr_accuracy:.4f}")
print(f"V1 XGBoost             : 0.5093")
print(f"V2 XGBoost             : {xgb_accuracy:.4f}")
print("=" * 40)
print(f"LR improvement  : {(lr_accuracy - 0.5211):.4f}")
print(f"XGB improvement : {(xgb_accuracy - 0.5093):.4f}")

      V1 vs V2 ACCURACY COMPARISON
V1 Logistic Regression : 0.5211
V2 Logistic Regression : 0.5124
V1 XGBoost             : 0.5093
V2 XGBoost             : 0.5483
LR improvement  : -0.0087
XGB improvement : 0.0390


## Save V2 Models

In [7]:
with open(os.path.join(BASE, 'Model_Training', 'xgb_model_v2.pkl'), 'wb') as f:
    pickle.dump(xgb_model, f)

with open(os.path.join(BASE, 'Model_Training', 'lr_model_v2.pkl'), 'wb') as f:
    pickle.dump(lr_model, f)

with open(os.path.join(BASE, 'Model_Training', 'scaler_v2.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

with open(os.path.join(BASE, 'Model_Training', 'label_encoder_v2.pkl'), 'wb') as f:
    pickle.dump(le, f)

print("V2 models saved!")

V2 models saved!
